In [11]:
import pandas as pd
import numpy as np
from datetime import datetime
import math
import re
import us
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import googlemaps
from geopy.geocoders import Nominatim
from dotenv import load_dotenv
import os

In [12]:
# Инициализация веб-драйвера
chrome_options = Options()
chrome_options.add_argument('--disable-gpu')

driver = webdriver.Chrome(options=chrome_options)

# URL для отправки запроса
url = "https://points.worldsdc.com/lookup2020"

max_attempts = 5  # Максимальное количество неудачных попыток
attempts = 0  # Счетчик неудачных попыток
dancer_id = 23000  # Номер танцора для начала проверки

while attempts < max_attempts:
    try:
        driver.get(url)
        input_field = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "q"))
        )
        input_field.clear()
        input_field.send_keys(str(dancer_id))
        form = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "lookup_form"))
        )
        form.submit()
        WebDriverWait(driver, 10).until(
            EC.text_to_be_present_in_element((By.ID, 'lookup_results'), str(dancer_id))
        )
        print(f"Информация найдена для танцора с номером: {dancer_id}")
        attempts = 0  # Обнуляем счетчик при успешной попытке
    except Exception as e:
        print(f"Для танцора с номером {dancer_id} информация не найдена.")
        attempts += 1  # Увеличиваем счетчик неудачных попыток
    dancer_id += 1  # Переходим к следующему номеру танцора

    # Если достигнуто максимальное количество неудачных попыток подряд, прерываем цикл
    if attempts == max_attempts:
        break

Информация найдена для танцора с номером: 23000
Информация найдена для танцора с номером: 23001
Информация найдена для танцора с номером: 23002
Информация найдена для танцора с номером: 23003
Информация найдена для танцора с номером: 23004
Информация найдена для танцора с номером: 23005
Информация найдена для танцора с номером: 23006
Информация найдена для танцора с номером: 23007
Информация найдена для танцора с номером: 23008
Информация найдена для танцора с номером: 23009
Информация найдена для танцора с номером: 23010
Информация найдена для танцора с номером: 23011
Информация найдена для танцора с номером: 23012
Для танцора с номером 23013 информация не найдена.
Для танцора с номером 23014 информация не найдена.
Для танцора с номером 23015 информация не найдена.
Для танцора с номером 23016 информация не найдена.
Для танцора с номером 23017 информация не найдена.


In [13]:
# у нас есть файл, который содержит сведения о значениях, где нет информации о танцоре, и парсить там нечего
def read_from_file(filename):
    with open(filename, 'r') as file:
        data = file.read().splitlines()
    return data

# Читаем список из файла
retrieved_empty_ids = read_from_file('empty_dancer_ids.txt')
retrieved_empty_ids = [int(id_str) for id_str in retrieved_empty_ids]

In [14]:
# Создаем список для хранения информации о каждом танцоре
soup_list = []

# Максимальное количество танцоров для обработки
max_dancer_id = dancer_id - 5

# пройдем циклом от 1 до последнего танцора, пропуская номера, под которыми танцоров в базе нет
for dancer_id in range(1, max_dancer_id + 1):
    if dancer_id in retrieved_empty_ids:
        continue
        # Переход на страницу
    driver.get(url)

    try:
        # Ожидание появления поля ввода для учетного номера танцора
        input_field = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "q"))
        )
        input_field.clear()  # Очистка поля ввода перед вводом нового номера
        input_field.send_keys(str(dancer_id))  # Ввод номера танцора

        # Находим форму с кнопкой "Найти танцора" и выполняем отправку
        form = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "lookup_form"))
        )
        form.submit()

        # Ждем появления элемента с нужным текстом внутри блока lookup_results
        element = WebDriverWait(driver, 10).until(
            EC.text_to_be_present_in_element((By.ID, 'lookup_results'), str(dancer_id))
        )

        # Получаем HTML-содержимое страницы после выполнения запроса
        html_content = driver.page_source

        soup = BeautifulSoup(html_content, "html.parser")
        
        # Добавляем soup в список для каждого танцора
        soup_list.append(soup)
        
    except Exception as e:
        print(f"Ошибка при обработке танцора {dancer_id}: {e}")       

# Закрываем драйвер после завершения цикла
if driver:
    driver.quit()

Ошибка при обработке танцора 4242: Message: timeout: Timed out receiving message from renderer: 300.000
  (Session info: chrome=121.0.6167.140)
Stacktrace:
	GetHandleVerifier [0x00007FF7AA085E42+3538674]
	(No symbol) [0x00007FF7A9CA4C02]
	(No symbol) [0x00007FF7A9B55AEB]
	(No symbol) [0x00007FF7A9B42FE8]
	(No symbol) [0x00007FF7A9B42E9F]
	(No symbol) [0x00007FF7A9B413A5]
	(No symbol) [0x00007FF7A9B41E5F]
	(No symbol) [0x00007FF7A9B4F32E]
	(No symbol) [0x00007FF7A9B61D83]
	(No symbol) [0x00007FF7A9B661DA]
	(No symbol) [0x00007FF7A9B42420]
	(No symbol) [0x00007FF7A9B61B9A]
	(No symbol) [0x00007FF7A9BDBCBA]
	(No symbol) [0x00007FF7A9BBEE53]
	(No symbol) [0x00007FF7A9B8F514]
	(No symbol) [0x00007FF7A9B90631]
	GetHandleVerifier [0x00007FF7AA0B6CAD+3738973]
	GetHandleVerifier [0x00007FF7AA10C506+4089270]
	GetHandleVerifier [0x00007FF7AA104823+4057299]
	GetHandleVerifier [0x00007FF7A9DD5C49+720121]
	(No symbol) [0x00007FF7A9CB126F]
	(No symbol) [0x00007FF7A9CAC304]
	(No symbol) [0x00007FF7A9C

TimeoutException: Message: timeout: Timed out receiving message from renderer: 300.000
  (Session info: chrome=121.0.6167.140)
Stacktrace:
	GetHandleVerifier [0x00007FF7AA085E42+3538674]
	(No symbol) [0x00007FF7A9CA4C02]
	(No symbol) [0x00007FF7A9B55AEB]
	(No symbol) [0x00007FF7A9B42FE8]
	(No symbol) [0x00007FF7A9B42E9F]
	(No symbol) [0x00007FF7A9B413A5]
	(No symbol) [0x00007FF7A9B41E5F]
	(No symbol) [0x00007FF7A9B4F32E]
	(No symbol) [0x00007FF7A9B61D83]
	(No symbol) [0x00007FF7A9B661DA]
	(No symbol) [0x00007FF7A9B42420]
	(No symbol) [0x00007FF7A9B61B9A]
	(No symbol) [0x00007FF7A9BDBCBA]
	(No symbol) [0x00007FF7A9BBEE53]
	(No symbol) [0x00007FF7A9B8F514]
	(No symbol) [0x00007FF7A9B90631]
	GetHandleVerifier [0x00007FF7AA0B6CAD+3738973]
	GetHandleVerifier [0x00007FF7AA10C506+4089270]
	GetHandleVerifier [0x00007FF7AA104823+4057299]
	GetHandleVerifier [0x00007FF7A9DD5C49+720121]
	(No symbol) [0x00007FF7A9CB126F]
	(No symbol) [0x00007FF7A9CAC304]
	(No symbol) [0x00007FF7A9CAC432]
	(No symbo

In [ ]:
# dancer_role_info = pd.read_csv('dancer_role_info.csv')
# dancers_points_info = pd.read_csv('dancers_points_info.csv')
# dancers_results_info = pd.read_csv('dancers_results_info.csv')

In [ ]:
def extract_role_info(soup, dancer_id):
    roles = {
        'dancer_id': dancer_id,
        'dancer_name': None,
        'primary_role': None,
        'primary_role_down_class': None,
        'primary_role_up_class': None,
        'secondary_role': None,
        'secondary_role_down_class': None,
        'secondary_role_up_class': None,
    }

    role_info = soup.find_all('p', class_='lead')

    for info in role_info:
        strong_tags = info.find_all('strong')
        for strong_tag in strong_tags:
            role_name = strong_tag.text.strip().replace(':', '')
            label_tags = strong_tag.find_next_siblings('span', class_='label')
            if label_tags:
                label_texts = []
                for tag in label_tags:
                    label_text = re.sub(r'\s+', ' ', tag.text.strip())
                    label_texts.append(label_text)
                if "Primary Role Leader Dance Level" in role_name:
                    roles['primary_role_down_class'] = label_texts[0]
                    roles['primary_role_up_class'] = label_texts[1] if len(label_texts) > 1 else np.nan
                    roles['primary_role'] = 'Leader'
                elif "Primary Role Follower Dance Level" in role_name:
                    roles['primary_role_down_class'] = label_texts[0]
                    roles['primary_role_up_class'] = label_texts[1] if len(label_texts) > 1 else np.nan
                    roles['primary_role'] = 'Follower'
                elif "Secondary Role Leader Dance Level" in role_name:
                    roles['secondary_role_down_class'] = label_texts[0]
                    roles['secondary_role_up_class'] = label_texts[1] if len(label_texts) > 1 else np.nan
                    roles['secondary_role'] = 'Leader'
                elif "Secondary Role Follower Dance Level" in role_name:
                    roles['secondary_role_down_class'] = label_texts[0]
                    roles['secondary_role_up_class'] = label_texts[1] if len(label_texts) > 1 else np.nan
                    roles['secondary_role'] = 'Follower'

    if roles['primary_role'] is None:  # Добавлено условие для танцоров без блока strong
        h2_tags = soup.find_all('h2')
        h3_tags = soup.find_all('h3')
        if h2_tags:
            h2_text_parts = h2_tags[-1].text.strip().split()
            roles['primary_role'] = h2_text_parts[-1] if h2_text_parts else np.nan
        if h3_tags:
            h3_text_parts = h3_tags[0].text.strip().split()
            roles['primary_role_down_class'] = h3_text_parts[0] if h3_text_parts else np.nan

    dancer_name = soup.find('h1').text.strip()
    roles['dancer_name'] = dancer_name

    return roles

# Создание словаря для быстрого доступа к soup по dancer_id
soup_dict = {}

for soup in soup_list:
    try:
        found_dancer_id = int(soup.find('h1').text.strip().split()[-1][1:-1])
        soup_dict[found_dancer_id] = soup
    except ValueError:
        pass

# Создание списка информации о танцорах
all_dancers_info = []

for dancer_id in range(1, max_dancer_id):
    if dancer_id in soup_dict:
        dancer_role_info = extract_role_info(soup_dict[dancer_id], dancer_id)
        all_dancers_info.append(dancer_role_info)
    else:
        roles = {'dancer_id': dancer_id}
        for key in ['dancer_name', 'primary_role', 'primary_role_down_class', 'primary_role_up_class',
                    'secondary_role', 'secondary_role_down_class', 'secondary_role_up_class']:
            roles[key] = np.nan
        all_dancers_info.append(roles)

dancer_role_info_df = pd.DataFrame(all_dancers_info)

# Упорядочивание столбцов
dancer_role_info = dancer_role_info_df[['dancer_id', 'dancer_name', 'primary_role', 'primary_role_down_class',
                                        'primary_role_up_class', 'secondary_role', 'secondary_role_down_class',
                                        'secondary_role_up_class']]

#Удаление технических символов и номера танцора из его имени
dancer_role_info['dancer_name'] = dancer_role_info['dancer_name'].str.replace('\t', '')
dancer_role_info['dancer_name'] = dancer_role_info['dancer_name'].str.replace(r'\s*\(\d+\)$', '', regex=True)






#работа с номерами, по которым в базе нет танцоров
empty_names = dancer_role_info[dancer_role_info['dancer_name'].isna()]
empty_dancer_ids = empty_names['dancer_id'].tolist()

# Функция для удаления дубликатов и сортировки строк в файле
def remove_duplicates_and_sort(filename):
    with open(filename, 'r') as file:
        lines = file.readlines()
    
    # Преобразование во множество для удаления дубликатов
    lines = list(set(lines))
    
    # Сортировка строк по возрастанию
    lines.sort(key=lambda x: int(x.strip()))

    with open(filename, 'w') as file:
        # Перезаписываем файл без дубликатов и отсортированный
        file.writelines(lines)

# Удаление дубликатов и сортировка строк в файле 'empty_dancer_ids.txt'
remove_duplicates_and_sort('empty_dancer_ids.txt')



#работа со столбцами и заменой значений
columns_to_replace = ['primary_role', 'primary_role_down_class', 'primary_role_up_class', 
                      'secondary_role', 'secondary_role_down_class', 'secondary_role_up_class']

# Создание словаря для замены значений
replacement_dict = {'NOV': 'Novice', 
                    'NEW': 'Newcomer',
                    'INT': 'Intermediate',
                    'ADV': 'Advanced',
                    'ALS': 'All-Star',
                    'CHMP': 'Champion',
                    'Champions': 'Champion',
                    'All-Stars': 'All-Star',
                    'Masters' : 'Master'}

# Замена значений и учет пропущенных значений (NaN)
dancer_role_info[columns_to_replace] = dancer_role_info[columns_to_replace].fillna('').replace(replacement_dict)

#удаление пустых строк и реиндексация данных
dancer_role_info.dropna(subset=['dancer_name'], inplace=True)
dancer_role_info.reset_index(drop=True, inplace=True)

In [ ]:
# # Загрузка предыдущей версии данных из файла 'dancer_role_info.csv'
# filename_previous = 'dancer_role_info.csv'
# if os.path.isfile(filename_previous):
#     saved_dancer_role_info = pd.read_csv(filename_previous)
# else:
#     saved_dancer_role_info = pd.DataFrame()

# # Загрузка предыдущего лога изменений из файла 'dancer_role_info_with_changes.csv'
# filename_logs = 'dancer_role_info_with_changes.csv'
# if os.path.isfile(filename_logs):
#     saved_dancer_role_info_with_changes = pd.read_csv(filename_logs)
# else:
#     saved_dancer_role_info_with_changes = pd.DataFrame()

# # Определение изменений на основе 'dancer_id' и даты апдейта
# mask = ~saved_dancer_role_info.apply(lambda row: (row['dancer_id']) in \
#                                      set(zip(dancer_role_info['dancer_id'])), axis=1)
# changed_rows = saved_dancer_role_info[mask]

# # Добавление временной метки и типа изменения
# current_time = datetime.now()
# changed_rows['change_type'] = 'Changed'
# changed_rows['last_updated'] = current_time

# # Обновление лога предыдущих изменений
# updated_dancer_role_info_with_changes = pd.concat([saved_dancer_role_info_with_changes, changed_rows])

# # Сохранение обновленного лога с изменениями
# updated_dancer_role_info_with_changes.to_csv(filename_logs, index=False)

In [ ]:
dancer_role_info

,dancer_id,dancer_name,primary_role,primary_role_down_class,primary_role_up_class,secondary_role,secondary_role_down_class,secondary_role_up_class
0,1,Diane Abele,Follower,Intermediate,,Leader,Novice,
1,2,Ellie Abrahams,Follower,Newcomer,Novice,Leader,Newcomer,Novice
2,3,Nathan Absher,Follower,Newcomer,Novice,Leader,Newcomer,Novice
3,4,James Adair,Leader,Advanced,,Follower,Intermediate,
4,5,Jodee Adair,Follower,Advanced,,,,
...,...,...,...,...,...,...,...,...
21910,22933,Shannon Larmore,Follower,Novice,,Leader,Newcomer,Novice
21911,22934,Vivian Yang,Follower,Novice,,Leader,Newcomer,Novice
21912,22935,Hima Kopally,Follower,Novice,,Leader,Newcomer,Novice
21913,22936,Josie Wiklund,Follower,Novice,,Leader,Newcomer,Novice


In [ ]:
dancer_role_info[dancer_role_info['dancer_name'].isna()]

,dancer_id,dancer_name,primary_role,primary_role_down_class,primary_role_up_class,secondary_role,secondary_role_down_class,secondary_role_up_class


In [ ]:
def extract_dancer_info(soup):
    dancer_info = []

    h1 = soup.find('h1')
    dancer_name = h1.text.strip()
    dancer_id_match = re.search(r'\((\d+)\)', dancer_name)  # Ищем цифры в кавычках
    if dancer_id_match:
        dancer_id = dancer_id_match.group(1)
    
    h2_elements = soup.find_all('h2')
    
    for h2 in h2_elements:
        dance = h2.text.strip().split('-')[0].strip()  # Оставляем только последнее слово в event_dance
        h2_text = h2.text.strip().split('-')[-1].strip()  # Оставляем только последнюю часть в role_dancer
        
        h3_elements = h2.find_all_next('h3')  # Находим все элементы h3 после текущего h2
        
        if h3_elements:
            for h3 in h3_elements:
                h3_text = h3.text.strip().split()[0]  # Оставляем только первую часть в class_dancer
                points = h3.find('span', class_='label').text.strip().split()[0]
                dancer_info.append([dancer_id, dance, h2_text, h3_text, points])
        else:
            dancer_info.append([dancer_id, dancer_name, dance, h2_text, None, None])
    
    columns = ['dancer_id', 'dance', 'role_dancer', 'class_dancer', 'points']
    dancer_df = pd.DataFrame(dancer_info, columns=columns)
    return dancer_df

# Пример использования для списка soup_list
dancers_points_info = pd.DataFrame()

for soup in soup_list:
    dancer_info_df = extract_dancer_info(soup)
    dancers_points_info = pd.concat([dancers_points_info, dancer_info_df], ignore_index=True)
    
dancers_points_info['points'] = dancers_points_info['points'].astype('int64')

In [ ]:
dancers_points_info.tail()

,dancer_id,dance,role_dancer,class_dancer,points
41239,22933,West Coast Swing,Follower,Newcomer,2
41240,22934,West Coast Swing,Follower,Newcomer,1
41241,22935,West Coast Swing,Follower,Newcomer,1
41242,22936,West Coast Swing,Follower,Novice,1
41243,22937,West Coast Swing,Follower,Novice,1


In [ ]:
def extract_dancer_results(soup):
    dancer_results = []

    h1 = soup.find('h1')
    dancer_id_match = re.search(r'\((\d+)\)', h1.text.strip())  # Находим цифры в кавычках
    if dancer_id_match:
        dancer_id = dancer_id_match.group(1)
    
    h2_elements = soup.find_all('h2')
    
    for h2 in h2_elements:
        event_dance = h2.text.strip().split('-')[0].strip()  # Оставляем только последнее слово в primary_role
        
        h3_elements = h2.find_all_next('h3')  # Находим все элементы h3 после текущего h2
        
        if h3_elements:
            for h3 in h3_elements:
                event_competition = h3.text.strip().split()[0]  # Оставляем только первую часть в class_dancer
                table = h3.find_next('table', class_='table-striped')
                if table:
                    rows = table.find_all('tr')
                    for row in rows[1:]:  # Начинаем с 1, чтобы пропустить заголовок таблицы
                        row_data = [dancer_id, event_dance, event_competition] + [cell.text.strip() for cell in row.find_all('td')]
                        dancer_results.append(row_data)
    
    columns = ['dancer_id', 'event_dance', 'event_competition', 'Role', 'Event', 'Location', 'Date', 'Result', 'Points']
    dancer_results_df = pd.DataFrame(dancer_results, columns=columns)
    return dancer_results_df

# Пример использования для списка soup_list
dancers_results_info = pd.DataFrame()

for soup in soup_list:
    dancer_results_df = extract_dancer_results(soup)
    dancers_results_info = pd.concat([dancers_results_info, dancer_results_df], ignore_index=True)

In [ ]:
dancers_results_info.rename(columns={'Role':'event_role', 
                                     'Event':'event_name', 
                                     'Location':'event_location', 
                                     'Date':'event_date', 
                                     'Result':'event_result', 
                                     'Points':'event_points'}, inplace=True)

In [ ]:
dancers_results_info.loc[dancers_results_info['event_name'] == 'Go West Swing Fest', 'event_location'] = 'Fremantle, Australia'
dancers_results_info.loc[dancers_results_info['event_location'] == 'Adelaide, South Australia, Australia', 'event_location'] = 'Adelaide, Australia'
dancers_results_info.loc[dancers_results_info['event_location'] == 'Budapest', 'event_location'] = 'Budapest, Hungary'
dancers_results_info.loc[dancers_results_info['event_location'] == 'Calgar Yy, Alberta', 'event_location'] = 'Calgary, Canada'
dancers_results_info.loc[dancers_results_info['event_location'] == 'Czech Republic', 'event_location'] = 'Brno, Czech Republic'
dancers_results_info.loc[dancers_results_info['event_location'] == 'Dallas, Texas', 'event_location'] = 'Dallas, TX'
dancers_results_info.loc[dancers_results_info['event_location'] == 'East Rutherford', 'event_location'] = 'East Rutherford, NJ'
dancers_results_info.loc[dancers_results_info['event_location'] == 'Edmonton, ON', 'event_location'] = 'Edmonton, Canada'
dancers_results_info.loc[dancers_results_info['event_location'] == 'Gold Coast, Queensland', 'event_location'] = 'Gold Coast, Australia'
dancers_results_info.loc[dancers_results_info['event_location'] == 'Israel', 'event_location'] = 'Tel Aviv, Israel'
dancers_results_info.loc[dancers_results_info['event_location'] == 'Ottawa', 'event_location'] = 'Ottawa, Canada'
dancers_results_info.loc[dancers_results_info['event_location'] == 'Paris', 'event_location'] = 'Paris, France'
dancers_results_info.loc[dancers_results_info['event_location'] == 'Sweden', 'event_location'] = 'Stockholm, Sweden'
dancers_results_info.loc[dancers_results_info['event_location'] == 'Toulouse', 'event_location'] = 'Toulouse, France'
dancers_results_info.loc[dancers_results_info['event_location'] == 'Redmond, Oregon', 'event_location'] = 'Redmond, OR'
dancers_results_info.loc[dancers_results_info['event_location'] == 'Seoul, South Korea', 'event_location'] = 'Seoul, Republic of Korea'
dancers_results_info.loc[dancers_results_info['event_location'] == 'Seoul, Korea', 'event_location'] = 'Seoul, Republic of Korea'
dancers_results_info.loc[dancers_results_info['event_location'] == 'Concord CA', 'event_location'] = 'Concord, CA'
dancers_results_info['event_location'] = dancers_results_info['event_location'].str.replace('Scotland', 'United Kingdom')
dancers_results_info['event_location'] = dancers_results_info['event_location'].str.replace('ENGLAND', 'United Kingdom')
dancers_results_info['event_location'] = dancers_results_info['event_location'].str.replace('England', 'United Kingdom')
dancers_results_info['event_location'] = dancers_results_info['event_location'].str.replace('UK', 'United Kingdom')
dancers_results_info['event_location'] = dancers_results_info['event_location'].str.replace('FRANCE', 'France')
dancers_results_info['event_location'] = dancers_results_info['event_location'].str.replace('QC Canada', 'Canada')
dancers_results_info['event_location'] = dancers_results_info['event_location'].str.replace('QC', 'Canada')
dancers_results_info['event_location'] = dancers_results_info['event_location'].str.replace('Isreal', 'Israel')
dancers_results_info['event_location'] = dancers_results_info['event_location'].str.replace('Washington Dc', 'Washington')
dancers_results_info['event_location'] = dancers_results_info['event_location'].str.replace('Kindom', 'Kingdom')
dancers_results_info['event_location'] = dancers_results_info['event_location'].str.replace('Italia', 'Italy')
dancers_results_info['event_location'] = dancers_results_info['event_location'].str.replace('BC', 'Canada')
dancers_results_info['event_location'] = dancers_results_info['event_location'].str.replace('Bernadino', 'Bernardino')
dancers_results_info['event_location'] = dancers_results_info['event_location'].str.replace('Minn / St. Paul', 'St. Paul')
dancers_results_info['event_location'] = dancers_results_info['event_location'].str.replace('St. Burlatskaya, Russia', 'Samara, Russia')

In [ ]:
dancers_results_info

,dancer_id,event_dance,event_competition,event_role,event_name,event_location,event_date,event_result,event_points
0,1,West Coast Swing,Intermediate,F,Monterey SwingFest,"Monterey, CA",January 1997,3,0
1,1,West Coast Swing,Novice,F,Boogie By The Bay,"San Francisco, CA",October 1994,4,3
2,1,West Coast Swing,Newcomer,F,4TH of July Convention,"Phoenix, AZ",July 1993,3,4
3,2,West Coast Swing,Masters,F,4TH of July Convention,"Phoenix, AZ",July 2003,F,1
4,2,West Coast Swing,Masters,F,4TH of July Convention,"Phoenix, AZ",July 1998,1,10
...,...,...,...,...,...,...,...,...,...
157515,22933,West Coast Swing,Newcomer,F,Freedom Swing Dance Challenge,"Philadelphia, PA",January 2024,5,2
157516,22934,West Coast Swing,Newcomer,F,Freedom Swing Dance Challenge,"Philadelphia, PA",January 2024,F,1
157517,22935,West Coast Swing,Newcomer,F,Freedom Swing Dance Challenge,"Philadelphia, PA",January 2024,F,1
157518,22936,West Coast Swing,Novice,F,Freedom Swing Dance Challenge,"Philadelphia, PA",January 2024,F,1


In [ ]:
unique_values = dancers_results_info['event_location'].unique()

# Создаем новый датафрейм с уникальными значениями
location_info = pd.DataFrame({'event_location': unique_values})

In [ ]:
# Загрузка переменных среды из файла .env
load_dotenv()

# Получение ключа API из переменной среды
api_key = os.getenv('GOOGLE_MAPS_API_KEY')

# Проверка, что ключ API получен успешно
if api_key:
    gmaps = googlemaps.Client(key=api_key)
    # Ваш код для работы с Google Maps API с использованием переменной gmaps
else:
    print("Ключ API не найден. Убедитесь, что переменная среды GOOGLE_MAPS_API_KEY установлена в файле .env.")

# Инициализация клиента Google Maps
gmaps = googlemaps.Client(key=api_key)

# Функция для получения координат по названию локации
def get_coordinates(location):
    try:
        # Получение геоданных от Google Maps Geocoding API
        geocode_result = gmaps.geocode(location)
        
        # Если есть результат, возвращаем координаты
        if geocode_result:
            return geocode_result[0]['geometry']['location']['lat'], geocode_result[0]['geometry']['location']['lng']
        else:
            return None
    except Exception as e:
        print(f"Error fetching coordinates for {location}: {e}")
        return None

# Применяем функцию к столбцу "event_location" и создаем новые столбцы "latitude" и "longitude"
location_info["Coordinates"] = location_info["event_location"].apply(get_coordinates)

# Разделяем координаты на два столбца
location_info[["latitude", "longitude"]] = location_info["Coordinates"].apply(pd.Series)

# Удаление временного столбца с координатами, если нужно
location_info.drop(columns=["Coordinates"], inplace=True)

In [ ]:
dancers_results_info.loc[dancers_results_info['event_name'] == 'Arizona Dance Classic (Cancelled due to Covid-19)', 'event_name'] = 'Arizona Dance Classic'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Asia WCS Open - 10th Anniversary', 'event_name'] = 'Asia WCS Open'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Asia West Coast Swing Open', 'event_name'] = 'Asia WCS Open'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Asian WCS Open Swingvitation', 'event_name'] = 'Asia WCS Open'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Bavarian Open West Coast Swing Championships', 'event_name'] = 'Bavarian Open'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Bavarian Open WCS', 'event_name'] = 'Bavarian Open'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Best of the Best WCS', 'event_name'] = 'Best of the Best'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Big Apple Dance Festival/World Hustle Championships', 'event_name'] = 'Big Apple Dance Festival'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Bridgetown Swing Boogie', 'event_name'] = 'BridgeTown Swing'
dancers_results_info.loc[dancers_results_info['event_name'] == 'C.A.S.H. Bash', 'event_name'] = 'C.A.S.H. Bash Weekend'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Canadian Swing Championships (CSC)', 'event_name'] = 'Canadian Swing Championships'
dancers_results_info.loc[dancers_results_info['event_name'] == "Capital Swing Presidents' Day Convention", 'event_name'] = 'Capital Swing Dance Convention'
dancers_results_info.loc[dancers_results_info['event_name'] == "Capital Swing Dancers' President's Day", 'event_name'] = 'Capital Swing Dance Convention'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Charlotte WestieFest', 'event_name'] = 'Charlotte Westie Fest'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Chicagoland', 'event_name'] = 'Chicagoland Dance Festival'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Chicagoland Country & Swing Dance Festival', 'event_name'] = 'Chicagoland Dance Festival'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Chicagoland Country and Swing Dance Festival', 'event_name'] = 'Chicagoland Dance Festival'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Citadel Swing (Cancelled due to Covid-19)', 'event_name'] = 'Citadel Swing'
dancers_results_info.loc[dancers_results_info['event_name'] == 'City of Angels Swing Event', 'event_name'] = 'City of Angels'
dancers_results_info.loc[dancers_results_info['event_name'] == 'D-Town Swing', 'event_name'] = 'D-Townswing'
dancers_results_info.loc[dancers_results_info['event_name'] == 'DC Swing Experience (DCSX)', 'event_name'] = 'DC Swing eXperience'
dancers_results_info.loc[dancers_results_info['event_name'] == 'DC Swing eXperience (DCSX)', 'event_name'] = 'DC Swing eXperience'
dancers_results_info.loc[dancers_results_info['event_name'] == 'DC Swing eXperience  (DCSX)', 'event_name'] = 'DC Swing eXperience'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Desert City Swing Dance Championships', 'event_name'] = 'Desert City Swing'
dancers_results_info.loc[dancers_results_info['event_name'] == 'DOWCS', 'event_name'] = 'Dutch Open'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Dutch Open West Coast Swing', 'event_name'] = 'Dutch Open'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Floor Play Swing Vacation', 'event_name'] = 'Floorplay New Years Swing Vacation'
dancers_results_info.loc[dancers_results_info['event_name'] == 'German Open WCS Championships', 'event_name'] = 'German Open'
dancers_results_info.loc[dancers_results_info['event_name'] == 'German Open Championships', 'event_name'] = 'German Open'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Halloween Swingthing', 'event_name'] = 'Halloween SwingThing'
dancers_results_info.loc[dancers_results_info['event_name'] == "Jack & Jill O'Rama", 'event_name'] = "J&J O'Rama"
dancers_results_info.loc[dancers_results_info['event_name'] == 'Korean Open Swing Dance Championships', 'event_name'] = 'Korean Open WCS Championsips'
dancers_results_info.loc[dancers_results_info['event_name'] == 'LoneStar Invitational', 'event_name'] = 'Lone Star Invitational'
dancers_results_info.loc[dancers_results_info['event_name'] == 'MADjam (Mid Atlantic Dance Jam)', 'event_name'] = 'MADjam'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Mid-Atlantic Dance Jam', 'event_name'] = 'MADjam'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Mid Atlantic Dance Jam (MADjam)', 'event_name'] = 'MADjam'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Meet Me in St. Louis Swing Dance Championships', 'event_name'] = 'Meet Me In St Louis'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Meet Me in St Louis Swing Dance Championships', 'event_name'] = 'Meet Me In St Louis'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Monterey SwingFest', 'event_name'] = 'Monterey Swing Fest'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Monterey Swingfest (Cancelled due to Covid-19)', 'event_name'] = 'Monterey Swing Fest'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Monterey Swingfest', 'event_name'] = 'Monterey Swing Fest'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Montreal WCS Fest', 'event_name'] = 'Montreal Westie Fest'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Moscow Westie Dance Fest', 'event_name'] = 'Moscow Westie Fest'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Mountain Magic Dance Convention', 'event_name'] = 'Mountain Magic'
dancers_results_info.loc[dancers_results_info['event_name'] == 'New Zealand Open Swing Dance Championships', 'event_name'] = 'New Zealand Open'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Palm Springs New Years Swing Dance Classic', 'event_name'] = 'Palm Springs New Year'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Palm Springs Summer Dance Camp Classic', 'event_name'] = 'Palm Springs Swing Dance Classic'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Palm Springs Summer Dance Classic', 'event_name'] = 'Palm Springs Swing Dance Classic'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Paradise Country Dance Festival', 'event_name'] = 'Paradise Dance Festival'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Paradise Country and Swing Dance Festival', 'event_name'] = 'Paradise Dance Festival'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Paradise Country & Swing Dance Festival', 'event_name'] = 'Paradise Dance Festival'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Paris Swing Classic', 'event_name'] = 'Paris Westie Fest'
dancers_results_info.loc[dancers_results_info['event_name'] == '4TH of July Convention', 'event_name'] = 'Phoenix 4th of July'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Rocky Mountain Five Dance (RM5)', 'event_name'] = 'Rocky Mountain 5 Dance'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Russian Open WCS Championship', 'event_name'] = 'Russian Open'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Russian Open WCS Championships', 'event_name'] = 'Russian Open'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Scandinavian Open WCS', 'event_name'] = 'Scandinavian Open'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Scandinavian Open WCS 2022', 'event_name'] = 'Scandinavian Open'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Scandinavian Open WCS "SNOW"', 'event_name'] = 'Scandinavian Open'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Sea Sun & Swing Camp', 'event_name'] = 'Sea Sun and Swing'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Sea To Sky - Seattle', 'event_name'] = 'Sea to Sky'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Sea to Sky Seattle WCS', 'event_name'] = 'Sea to Sky'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Show-Me Showdown', 'event_name'] = 'Show Me Showdown'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Simply Adelaide West Coast Swing 2023', 'event_name'] = 'Simply Adelaide West Coast Swing'
dancers_results_info.loc[dancers_results_info['event_name'] == 'Simply Adelaide West Coast Swing 2022', 'event_name'] = 'Simply Adelaide West Coast Swing'
dancers_results_info.loc[dancers_results_info['event_name'] == 'SOswing 2022', 'event_name'] = 'SOswing'
dancers_results_info.loc[dancers_results_info['event_name'] == 'South Bay CW Dance Festival', 'event_name'] = 'South Bay Dance Fling'
dancers_results_info.loc[dancers_results_info['event_name'] == "Spotlight New Year's Celebration", 'event_name'] = 'Spotlight Dance Challenge'
dancers_results_info.loc[dancers_results_info['event_name'] == "Swing Over Orlando", 'event_name'] = 'Swing Over'
dancers_results_info.loc[dancers_results_info['event_name'] == "Swing&Snow", 'event_name'] = 'Swing & Snow'
dancers_results_info.loc[dancers_results_info['event_name'] == "SwingCouver 2020 - The 10th Episode", 'event_name'] = 'SwingCouver'
dancers_results_info.loc[dancers_results_info['event_name'] == "SwingDiego (The Superbowl of Swing)", 'event_name'] = 'SwingDiego'
dancers_results_info.loc[dancers_results_info['event_name'] == "Swingin' New England Dance Festival", 'event_name'] = "Swingin' New England"
dancers_results_info.loc[dancers_results_info['event_name'] == "Swingtacular: The Galactic Open 2023", 'event_name'] = 'Swingtacular'
dancers_results_info.loc[dancers_results_info['event_name'] == "Swingtacular: The Galactic Open 2022", 'event_name'] = 'Swingtacular'
dancers_results_info.loc[dancers_results_info['event_name'] == "Swingtacular: The Galactic Open", 'event_name'] = 'Swingtacular'
dancers_results_info.loc[dancers_results_info['event_name'] == "Swingtimate 2020", 'event_name'] = 'Swingtimate'
dancers_results_info.loc[dancers_results_info['event_name'] == "SwingTime Denver", 'event_name'] = 'SwingTime'
dancers_results_info.loc[dancers_results_info['event_name'] == "Swingtime in the Rockies", 'event_name'] = 'SwingTime'
dancers_results_info.loc[dancers_results_info['event_name'] == 'The After Party “TAP”', 'event_name'] = 'The After Party'
dancers_results_info.loc[dancers_results_info['event_name'] == "The After Party - TAP", 'event_name'] = 'The After Party'
dancers_results_info.loc[dancers_results_info['event_name'] == "The After Party (TAP)", 'event_name'] = 'The After Party'
dancers_results_info.loc[dancers_results_info['event_name'] == "The Brazilian Open Championships (Cancelled due to Coronavirus)", 'event_name'] = 'The Brazilian Open'
dancers_results_info.loc[dancers_results_info['event_name'] == "The Brazilian Open Championships", 'event_name'] = 'The Brazilian Open'
dancers_results_info.loc[dancers_results_info['event_name'] == "Toronto Open Swing & Hustle Championships", 'event_name'] = 'Toronto Open'
dancers_results_info.loc[dancers_results_info['event_name'] == "Toronto Open Swing and Hustle Championships", 'event_name'] = 'Toronto Open'
dancers_results_info.loc[dancers_results_info['event_name'] == "U.K. & European WCS Championships", 'event_name'] = 'UK & European WCS Championships'
dancers_results_info.loc[dancers_results_info['event_name'] == "USA Grand Nationals Dance Championships", 'event_name'] = 'USA Grand Nationals'
dancers_results_info.loc[dancers_results_info['event_name'] == "USA Grand National Dance Championships", 'event_name'] = 'USA Grand Nationals'
dancers_results_info.loc[dancers_results_info['event_name'] == "Winter White WCS", 'event_name'] = 'Winter White'
dancers_results_info.loc[dancers_results_info['event_name'] == "Winter White West Coast Swing", 'event_name'] = 'Winter White'
dancers_results_info.loc[dancers_results_info['event_name'] == "Toronto Open Swing & Hustle Championships", 'event_name'] = 'Toronto Open'
dancers_results_info.loc[dancers_results_info['event_name'] == "Toronto Open Swing & Hustle Championships", 'event_name'] = 'Toronto Open'
dancers_results_info.loc[dancers_results_info['event_name'] == "The New Zealand Open", 'event_name'] = 'New Zealand Open' 
dancers_results_info.loc[dancers_results_info['event_name'] == "Seattle's Easter Swing", 'event_name'] = 'Easter Swing'
dancers_results_info.loc[dancers_results_info['event_name'] == "Swing Trilogy", 'event_name'] = 'Trilogy Swing'
dancers_results_info.loc[dancers_results_info['event_name'] == "Rock the Barn", 'event_name'] = 'Rock The Barn' 
dancers_results_info.loc[dancers_results_info['event_name'] == "Texas Classic", 'event_name'] = 'The Texas Classic'
dancers_results_info.loc[dancers_results_info['event_name'] == "Westies on the Water", 'event_name'] = 'Westies on The Water'
dancers_results_info.loc[dancers_results_info['event_name'] == "WESTY NANTES", 'event_name'] = 'Westy Nantes'
dancers_results_info.loc[dancers_results_info['event_name'] == "Country Dance World Championships", 'event_name'] = 'Worlds UCWDC'
dancers_results_info.loc[dancers_results_info['event_name'] == "UCWDC Country Dance World Championships", 'event_name'] = 'Worlds UCWDC'
dancers_results_info.loc[dancers_results_info['event_name'] == "New Orleans Dance Mardi Gras", 'event_name'] = 'Dance Mardi Gras'
dancers_results_info.loc[dancers_results_info['event_name'] == "The Boston Tea Party", 'event_name'] = 'Boston Tea Party'
dancers_results_info.loc[dancers_results_info['event_name'] == "Floorplay New Years Swing Vacation", 'event_name'] = 'FloorPlay New Years Swing Vacation'
dancers_results_info.loc[dancers_results_info['event_name'] == "Swingover", 'event_name'] = 'Swing Over'
dancers_results_info.loc[dancers_results_info['event_name'] == "Boogie by the Bay", 'event_name'] = 'Boogie By The Bay'
dancers_results_info.loc[dancers_results_info['event_name'] == "Saint Petersburg WCS Nights", 'event_name'] = 'St.Petersburg WCS Nights'
dancers_results_info.loc[dancers_results_info['event_name'] == "Sweden Westie Gala", 'event_name'] = 'Westie Gala'
dancers_results_info.loc[dancers_results_info['event_name'] == "CASH Bash", 'event_name'] = 'C.A.S.H. Bash Weekend'
dancers_results_info.loc[dancers_results_info['event_name'] == "London SWINGvitational", 'event_name'] = 'London SwingVitational'

In [ ]:
# Функция для извлечения штата из строки
def extract_state(location):
    parts = location.split(', ')
    if len(parts) == 2 and len(parts[1]) == 2:
        return parts[1]
    else:
        return np.nan

# Функция для извлечения страны из строки
def extract_country(location):
    parts = location.split(', ')
    if len(parts) == 2 and len(parts[1]) == 2:
        return 'United States'
    elif len(parts) == 2:
        return parts[1]
    else:
        return parts[-1]

# Создание новых столбцов на основе функций extract_state и extract_country
location_info['event_state'] = location_info['event_location'].apply(extract_state)
location_info['event_country'] = location_info['event_location'].apply(extract_country)

In [ ]:
def replace_state(state_abbr):
    if state_abbr and not isinstance(state_abbr, float):
        if state_abbr == 'DC':  # Проверка для округа Колумбия
            return 'District of Columbia'
        state = us.states.lookup(state_abbr)
        return state.name if state else np.nan
    return np.nan

# Применение функции к столбцу 'State'
location_info['event_state'] = location_info['event_state'].apply(replace_state)

In [ ]:
def extract_city(location):
    parts = location.split(', ')
    return parts[0]

# Создание нового столбца 'City' на основе функции extract_city
location_info['event_city'] = location_info['event_location'].apply(extract_city)

In [ ]:
location_info['location_id'] = location_info.reset_index().index + 1

In [ ]:
def extract_month(location):
    parts = location.split(' ')
    return parts[0]

def extract_year(location):
    parts = location.split(' ')
    return parts[1]

# Создание нового столбца 'month' на основе функции extract_month
dancers_results_info['event_month'] = dancers_results_info['event_date'].apply(extract_month)
# Создание нового столбца 'year' на основе функции extract_year
dancers_results_info['event_year'] = dancers_results_info['event_date'].apply(extract_year)


In [ ]:
dancers_results_info = dancers_results_info.merge(location_info[['event_location', 'location_id']], on='event_location', how='left')

In [ ]:
dancers_results_info['event_role'] = dancers_results_info['event_role'].str.replace('F', 'Follower')
dancers_results_info['event_role'] = dancers_results_info['event_role'].str.replace('L', 'Leader')

In [ ]:
dancers_results_info

,dancer_id,event_dance,event_competition,event_role,event_name,event_location,event_date,event_result,event_points,event_month,event_year,location_id
0,1,West Coast Swing,Intermediate,Follower,Monterey Swing Fest,"Monterey, CA",January 1997,3,0,January,1997,1
1,1,West Coast Swing,Novice,Follower,Boogie By The Bay,"San Francisco, CA",October 1994,4,3,October,1994,2
2,1,West Coast Swing,Newcomer,Follower,Phoenix 4th of July,"Phoenix, AZ",July 1993,3,4,July,1993,3
3,2,West Coast Swing,Masters,Follower,Phoenix 4th of July,"Phoenix, AZ",July 2003,F,1,July,2003,3
4,2,West Coast Swing,Masters,Follower,Phoenix 4th of July,"Phoenix, AZ",July 1998,1,10,July,1998,3
...,...,...,...,...,...,...,...,...,...,...,...,...
157515,22933,West Coast Swing,Newcomer,Follower,Freedom Swing Dance Challenge,"Philadelphia, PA",January 2024,5,2,January,2024,79
157516,22934,West Coast Swing,Newcomer,Follower,Freedom Swing Dance Challenge,"Philadelphia, PA",January 2024,F,1,January,2024,79
157517,22935,West Coast Swing,Newcomer,Follower,Freedom Swing Dance Challenge,"Philadelphia, PA",January 2024,F,1,January,2024,79
157518,22936,West Coast Swing,Novice,Follower,Freedom Swing Dance Challenge,"Philadelphia, PA",January 2024,F,1,January,2024,79


In [ ]:
month_dict = {
    'January': 1,
    'February': 2,
    'March': 3,
    'April': 4,
    'May': 5,
    'June': 6,
    'July': 7,
    'August': 8,
    'September': 9,
    'October': 10,
    'November': 11,
    'December': 12,
    }

dancers_results_info['event_month'] = dancers_results_info['event_month'].map(month_dict)
dancers_results_info['event_month'] = pd.to_datetime(dancers_results_info['event_month'], format='%m')
dancers_results_info['event_month'] = dancers_results_info['event_month'].dt.month.astype('int64')
dancers_results_info['event_year'] = pd.to_datetime(dancers_results_info['event_year'], format='%Y')
dancers_results_info['event_year'] = dancers_results_info['event_year'].dt.strftime('%Y').astype('int64')
dancers_results_info['event_points'] = dancers_results_info['event_points'].astype('int64')
dancers_results_info['event_year_and_month'] = pd.to_datetime(dancers_results_info['event_year'].astype(str) + '-' + dancers_results_info['event_month'].astype(str), format='%Y-%m')


In [ ]:
dancers_results_info.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 157520 entries, 0 to 157519
Data columns (total 13 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   dancer_id             157520 non-null  object        
 1   event_dance           157520 non-null  object        
 2   event_competition     157520 non-null  object        
 3   event_role            157520 non-null  object        
 4   event_name            157520 non-null  object        
 5   event_location        157520 non-null  object        
 6   event_date            157520 non-null  object        
 7   event_result          157520 non-null  object        
 8   event_points          157520 non-null  int64         
 9   event_month           157520 non-null  int64         
 10  event_year            157520 non-null  int64         
 11  location_id           157520 non-null  int64         
 12  event_year_and_month  157520 non-null  datetime64[ns]
dtyp

In [ ]:
location_info

,event_location,latitude,longitude,event_state,event_country,event_city,location_id
0,"Monterey, CA",36.600238,-121.894676,California,United States,Monterey,1
1,"San Francisco, CA",37.774929,-122.419415,California,United States,San Francisco,2
2,"Phoenix, AZ",33.448377,-112.074037,Arizona,United States,Phoenix,3
3,"Portland, OR",45.515232,-122.678385,Oregon,United States,Portland,4
4,"Fresno, CA",36.737798,-119.787125,California,United States,Fresno,5
...,...,...,...,...,...,...,...
169,"Kazan, Russia",55.787894,49.123329,NaN,Russia,Kazan,170
170,"Samara, Russia",53.203772,50.160638,NaN,Russia,Samara,171
171,"Krakow, Poland",50.064650,19.944980,NaN,Poland,Krakow,172
172,"Kuopio, Finland",62.897970,27.678172,NaN,Finland,Kuopio,173


In [ ]:
# Удаление столбца Location, если он больше не нужен
location_info = location_info.reindex(columns=['location_id', 
                                                'event_city', 
                                                'event_state',
                                                'event_country',
                                                'latitude',
                                                'longitude',
                                                'event_location'
                                                ])



In [ ]:
# Удаление столбца Location, если он больше не нужен
dancers_results_info.drop(columns=['event_location', 'event_date'], inplace=True)  # Раскомментируйте, если необходимо удалить исходный столбец
dancers_results_info = dancers_results_info.reindex(columns=['dancer_id', 
                                                            'event_dance', 
                                                            'event_competition',
                                                            'event_role',
                                                            'event_result',
                                                            'event_points',
                                                            'event_name',
                                                            'location_id',
                                                            'event_year',
                                                            'event_month',
                                                            'event_year_and_month'
                                                            ])

In [ ]:
# Теперь надо удалить непреднамеренные дубли, которые возникли у танцоров, выступавших на одном турнире в 2 ролях
columns_to_check = dancers_results_info.columns.difference(['event_dance'])

# Удаление дубликатов на основе последних восьми столбцов
dancers_results_info = dancers_results_info.drop_duplicates(subset=columns_to_check, keep='last').reset_index(drop=True)

In [ ]:
columns_to_check

Index(['dancer_id', 'event_competition', 'event_month', 'event_name',
       'event_points', 'event_result', 'event_role', 'event_year',
       'event_year_and_month', 'location_id'],
      dtype='object')

In [ ]:
dancers_results_info

,dancer_id,event_dance,event_competition,event_role,event_result,event_points,event_name,location_id,event_year,event_month,event_year_and_month
0,1,West Coast Swing,Intermediate,Follower,3,0,Monterey Swing Fest,1,1997,1,1997-01-01
1,1,West Coast Swing,Novice,Follower,4,3,Boogie By The Bay,2,1994,10,1994-10-01
2,1,West Coast Swing,Newcomer,Follower,3,4,Phoenix 4th of July,3,1993,7,1993-07-01
3,2,West Coast Swing,Masters,Follower,F,1,Phoenix 4th of July,3,2003,7,2003-07-01
4,2,West Coast Swing,Masters,Follower,1,10,Phoenix 4th of July,3,1998,7,1998-07-01
...,...,...,...,...,...,...,...,...,...,...,...
154845,22933,West Coast Swing,Newcomer,Follower,5,2,Freedom Swing Dance Challenge,79,2024,1,2024-01-01
154846,22934,West Coast Swing,Newcomer,Follower,F,1,Freedom Swing Dance Challenge,79,2024,1,2024-01-01
154847,22935,West Coast Swing,Newcomer,Follower,F,1,Freedom Swing Dance Challenge,79,2024,1,2024-01-01
154848,22936,West Coast Swing,Novice,Follower,F,1,Freedom Swing Dance Challenge,79,2024,1,2024-01-01


In [ ]:
#Теперь выявляем изменения по сравнению с предыдущей выгрузкой dancer_role_info

# Загрузка предыдущего CSV файла 
try:
    old_dancer_role_info = pd.read_csv('dancer_role_info.csv')
except FileNotFoundError:
    old_dancer_role_info = pd.DataFrame()
    
old_dancer_role_info = old_dancer_role_info.fillna('')
    
# Добавление столбца с датой формирования версии к текущей версии dancer_role_info
dancer_role_info['update_date'] = datetime.now().strftime("%Y-%m-%d")

# Объединение старого и нового DataFrame
changed_dancer_role_info = pd.concat([old_dancer_role_info, dancer_role_info], ignore_index=True)

# Удаление дубликатов (исключая столбец с датой формирования версии)
changed_dancer_role_info = changed_dancer_role_info.drop_duplicates(subset=changed_dancer_role_info.columns.difference(['update_date']), keep=False)
# Сортировка по dancer_id
changed_dancer_role_info = changed_dancer_role_info.sort_values(by=['dancer_id', 'update_date']).reset_index(drop=True)

# Загрузка предыдущего CSV-файла
try:
    previous_df = pd.read_csv('changed_dancer_role_info.csv')
except FileNotFoundError:
    previous_df = pd.DataFrame()

# Объединение предыдущего DataFrame с текущими изменениями
combined_result_df = pd.concat([previous_df, changed_dancer_role_info], ignore_index=True)

In [ ]:
combined_result_df.tail(35)

,dancer_id,dancer_name,primary_role,primary_role_down_class,primary_role_up_class,secondary_role,secondary_role_down_class,secondary_role_up_class,update_date
193,22902,Gabriel Hotchkiss,Leader,Novice,,Follower,Newcomer,Novice,2024-01-26
194,22903,Nathan Wong,Leader,Novice,,Follower,Newcomer,Novice,2024-01-26
195,22904,Clay Morrison,Leader,Novice,,Follower,Newcomer,Novice,2024-01-26
196,22905,Austin Rains,Leader,Novice,,Follower,Newcomer,Novice,2024-01-26
197,22906,Madison Broussard,Follower,Novice,,Leader,Newcomer,Novice,2024-01-26
198,22908,Courtney Wolf,Follower,Novice,,Leader,Newcomer,Novice,2024-01-26
199,22909,Oksana Collins,Follower,Newcomer,Novice,Leader,Newcomer,Novice,2024-01-26
200,22910,Paul Parquet,Leader,Novice,,Follower,Newcomer,Novice,2024-01-26
201,22911,Gloria Lopez,Follower,Novice,,Leader,Newcomer,Novice,2024-01-26
202,22912,Marie Thienphoun,Follower,Novice,,Leader,Newcomer,Novice,2024-01-26


In [ ]:
#Теперь выявляем изменения по сравнению с предыдущей выгрузкой dancers_points_info

# Загрузка предыдущего CSV файла 
try:
    old_dancers_points_info = pd.read_csv('dancers_points_info.csv')
except FileNotFoundError:
    old_dancers_points_info = pd.DataFrame()

old_dancers_points_info = old_dancers_points_info.fillna('')
    
# Добавление столбца с датой формирования версии к текущей версии dancers_points_info
dancers_points_info['update_date'] = datetime.now().strftime("%Y-%m-%d")

# Объединение старого и нового DataFrame
changed_dancers_points_info = pd.concat([old_dancers_points_info, dancers_points_info], ignore_index=True)

changed_dancers_points_info['dancer_id'] = changed_dancers_points_info['dancer_id'].astype('int64')

# Удаление дубликатов (исключая столбец с датой формирования версии)
changed_dancers_points_info = changed_dancers_points_info.drop_duplicates(subset=changed_dancers_points_info.columns.difference(['update_date']), keep=False)
# Сортировка по dancer_id
changed_dancers_points_info = changed_dancers_points_info.sort_values(by=['dancer_id', 'class_dancer', 'update_date']).reset_index(drop=True)

# Загрузка предыдущего CSV-файла
try:
    previous_points = pd.read_csv('changed_dancer_points_info.csv')
except FileNotFoundError:
    previous_points = pd.DataFrame()

# Объединение предыдущего DataFrame с текущими изменениями
combined_points_df = pd.concat([previous_points, changed_dancers_points_info], ignore_index=True)

In [ ]:
combined_points_df.head(35)

,dancer_id,dance,role_dancer,class_dancer,points,update_date
0,216,West Coast Swing,Follower,Masters,372,2024-01-16
1,216,West Coast Swing,Follower,Masters,378,2024-01-26
2,320,West Coast Swing,Leader,Sophisticated,1,2024-01-26
3,820,West Coast Swing,Follower,Masters,483,2024-01-16
4,820,West Coast Swing,Follower,Masters,487,2024-01-26
5,1047,West Coast Swing,Leader,All-Stars,90,2024-01-16
6,1047,West Coast Swing,Leader,All-Stars,91,2024-01-26
7,1054,West Coast Swing,Follower,Masters,150,2024-01-16
8,1054,West Coast Swing,Follower,Masters,151,2024-01-26
9,1333,West Coast Swing,Follower,All-Stars,24,2024-01-16


In [ ]:
dancer_role_info.to_csv('dancer_role_info.csv', index=False)
dancers_points_info.to_csv('dancers_points_info.csv', index=False)
dancers_results_info.to_csv('dancers_results_info.csv', index=False)
location_info.to_csv('location_info.csv', index=False)
combined_result_df.to_csv('changed_dancer_role_info.csv', index=False)
combined_points_df.to_csv('changed_dancer_points_info.csv', index=False)